In [1]:
import numpy as numpy
import pandas as pd
import joblib as jb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv("Solar Power Plant Data.csv")
print(df.head(5))

           DateTime  WindSpeed  Sunshine  AirPressure  Radiation  \
0  01.01.2017-00:00        0.6         0       1003.8       -7.4   
1  01.01.2017-01:00        1.7         0       1003.5       -7.4   
2  01.01.2017-02:00        0.6         0       1003.4       -6.7   
3  01.01.2017-03:00        2.4         0       1003.3       -7.2   
4  01.01.2017-04:00        4.0         0       1003.1       -6.3   

   AirTemperature  RelativeAirHumidity  SystemProduction  
0             0.1                   97               0.0  
1            -0.2                   98               0.0  
2            -1.2                   99               0.0  
3            -1.3                   99               0.0  
4             3.6                   67               0.0  


In [3]:
print(df.dtypes)

DateTime                   str
WindSpeed              float64
Sunshine                 int64
AirPressure            float64
Radiation              float64
AirTemperature         float64
RelativeAirHumidity      int64
SystemProduction       float64
dtype: object


In [4]:
date = pd.DatetimeIndex(df["DateTime"])
df["Year"] = date.year
df["Month"] = date.month
df["Day"] = date.day
df["Time"] = date.time
df = df.drop(columns= "DateTime")
print(df.head(5))

   WindSpeed  Sunshine  AirPressure  Radiation  AirTemperature  \
0        0.6         0       1003.8       -7.4             0.1   
1        1.7         0       1003.5       -7.4            -0.2   
2        0.6         0       1003.4       -6.7            -1.2   
3        2.4         0       1003.3       -7.2            -1.3   
4        4.0         0       1003.1       -6.3             3.6   

   RelativeAirHumidity  SystemProduction  Year  Month  Day      Time  
0                   97               0.0  2017      1    1  00:00:00  
1                   98               0.0  2017      1    1  01:00:00  
2                   99               0.0  2017      1    1  02:00:00  
3                   99               0.0  2017      1    1  03:00:00  
4                   67               0.0  2017      1    1  04:00:00  


In [5]:
data = df["Time"]
data = pd.get_dummies(data).astype(int)
print(data.head(5))

   00:00:00  01:00:00  02:00:00  03:00:00  04:00:00  05:00:00  06:00:00  \
0         1         0         0         0         0         0         0   
1         0         1         0         0         0         0         0   
2         0         0         1         0         0         0         0   
3         0         0         0         1         0         0         0   
4         0         0         0         0         1         0         0   

   07:00:00  08:00:00  09:00:00  ...  14:00:00  15:00:00  16:00:00  17:00:00  \
0         0         0         0  ...         0         0         0         0   
1         0         0         0  ...         0         0         0         0   
2         0         0         0  ...         0         0         0         0   
3         0         0         0  ...         0         0         0         0   
4         0         0         0  ...         0         0         0         0   

   18:00:00  19:00:00  20:00:00  21:00:00  22:00:00  23:00:00  
0   

In [6]:
df = df.drop(columns= ["Time"])
df = pd.concat([df, data], axis=1).astype(int)
print(df.head(5))

   WindSpeed  Sunshine  AirPressure  Radiation  AirTemperature  \
0          0         0         1003         -7               0   
1          1         0         1003         -7               0   
2          0         0         1003         -6              -1   
3          2         0         1003         -7              -1   
4          4         0         1003         -6               3   

   RelativeAirHumidity  SystemProduction  Year  Month  Day  ...  14:00:00  \
0                   97                 0  2017      1    1  ...         0   
1                   98                 0  2017      1    1  ...         0   
2                   99                 0  2017      1    1  ...         0   
3                   99                 0  2017      1    1  ...         0   
4                   67                 0  2017      1    1  ...         0   

   15:00:00  16:00:00  17:00:00  18:00:00  19:00:00  20:00:00  21:00:00  \
0         0         0         0         0         0         0    

In [7]:
df.columns = df.columns.astype(str)

In [8]:
X = df.drop(columns= "SystemProduction")
y = df["SystemProduction"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

In [10]:
param_grid={
    "n_estimators":[100, 200, 300],
    "min_samples_split":[2, 5, 10, 20, 50],
    "min_samples_leaf":[2, 5, 10, 20, 50],
    "max_depth":[2, 5, 10, 50, 100]
}

In [11]:
model = GradientBoostingRegressor(random_state =42)

In [ ]:
grid_search = GridSearchCV(
    estimator = model,
    param_grid = param_grid,
    cv = 2,
    n_jobs = -1
)

grid_search.fit(X_train, y_train)

In [28]:
best_score_train = grid_search.best_score_
best_pramas_train = grid_search.best_params_
print(best_score_train)
print(best_pramas_train)

0.820367115407794
{'max_depth': 5, 'min_samples_leaf': 5, 'min_samples_split': 20, 'n_estimators': 200}


In [29]:
best_model = grid_search.best_estimator_
train_score = best_model.score(X_train, y_train)
test_score = best_model.score(X_test, y_test)
print("Train:", train_score)
print("Test:", test_score)

Train: 0.9538178504544146
Test: 0.8364256037240738


In [ ]:
jb.dump("solar_model.pkl")